In [ ]:
# Blessing Anoroh
#  Week Ten - Assignment: Document Classification
# Dat 620 - Web Analytics
# Date: April 20, 2026 (Due)

# Instructions
# It can be useful to be able to classify new "test" documents using already classified "training" documents.
# A common example is using a corpus of labeled spam and ham (non-spam) e-mails to predict whether or not a new document is spam.
# Here is one example of such data:  UCI Machine Learning Repository: Spambase Data Set
# For this project, you can either use the above dataset to predict the class of new documents (either withheld from the training dataset or from another source such as your own spam folder).
# For more adventurous students, you are welcome (encouraged!) to come up a different set of documents (including scraped web pages!?) that have already been classified (e.g. tagged),
# then analyze these documents to predict how new documents should be classified.


## Introduction

In this project, I worked with the **Spambase dataset** to classify emails as spam or not spam using machine learning. The dataset represents emails with numerical features such as word and character frequencies, making it suitable for classification without additional text processing.

This analysis focuses on preparing the data, building and comparing two models: **Logistic Regression** and **Naive Bayes** and evaluating how well they identify spam. The goal is to determine which model performs better and to understand how effectively machine learning can be used for spam detection.


In [ ]:
# Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Load dataset, from source
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data"

# Column names (from dataset description)
columns = [f'feature_{i}' for i in range(57)] + ['target']

data = pd.read_csv(url, header=None, names=columns)

# Preview data of first 5 rows
print("First 5 rows of the dataset:")
print(data.head())



First 5 rows of the dataset:
   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
0       0.00       0.64       0.64        0.0       0.32       0.00   
1       0.21       0.28       0.50        0.0       0.14       0.28   
2       0.06       0.00       0.71        0.0       1.23       0.19   
3       0.00       0.00       0.00        0.0       0.63       0.00   
4       0.00       0.00       0.00        0.0       0.63       0.00   

   feature_6  feature_7  feature_8  feature_9  ...  feature_48  feature_49  \
0       0.00       0.00       0.00       0.00  ...        0.00       0.000   
1       0.21       0.07       0.00       0.94  ...        0.00       0.132   
2       0.19       0.12       0.64       0.25  ...        0.01       0.143   
3       0.31       0.63       0.31       0.63  ...        0.00       0.137   
4       0.31       0.63       0.31       0.63  ...        0.00       0.135   

   feature_50  feature_51  feature_52  feature_53  feature_54  feature_55  

The dataset contains 57 feature columns and 1 target column, where the target indicates whether an email is spam(1) or not spam(0).

All features are numerical, so no text preprocessing is needed. This tells us with confirmation that the data is loaded correctly and ready for analyzation.

In [ ]:
# Data Checks and Cleaning

print("\nDataset shape before cleaning:", data.shape)

# Check missing values
if data.isnull().sum().sum() == 0:
    print("No missing values in the dataset")
else:
    print("Missing values found")

# Check duplicates
print("\nNumber of duplicate rows:", data.duplicated().sum())

# Remove duplicates if any exist
data = data.drop_duplicates()

print("\nDataset shape after removing duplicates:", data.shape)

# Check target class balance
print("\nTarget class distribution:")
print(data['target'].value_counts())
print("\nTarget class proportions:")
print(data['target'].value_counts(normalize=True))

# Split features and target
X = data.drop('target', axis=1)
y = data['target']

# Train-test split with stratify to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



Dataset shape before cleaning: (4601, 58)
No missing values in the dataset

Number of duplicate rows: 391

Dataset shape after removing duplicates: (4210, 58)

Target class distribution:
target
0    2531
1    1679
Name: count, dtype: int64

Target class proportions:
target
0    0.601188
1    0.398812
Name: proportion, dtype: float64


There are no missing values in the dataset. After checking the data, 391 duplicate rows were found and removed. This reduced the dataset size from 4601 rows to 4210 rows, while keeping the same 58 columns.

The target class distribution shows that:

*   2531 emails are labeled as 0 (not spam)
*   1679 emails are labeled as 1 (spam)



This means that about 60.1% of the emails are not spam, while 39.9% are spam. The dataset is slightly imbalanced, but not severely, so it is still suitable for training classification models without major adjustments.

Total: 2531 + 1679=4210 <br>

Spam: 1679 / 4210 * 100 = 39.9% <br>

Not Spam: 2531 / 4210 * 100 = 60.1%

In [ ]:
# Feature Scaling for Logistic Regression
# Logistic Regression performs better when features are scaled
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Model # 1: Logistic Regression
lr_model = LogisticRegression(max_iter=5000)
lr_model.fit(X_train_scaled, y_train)

lr_preds = lr_model.predict(X_test_scaled)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]

print("\n--- Logistic Regression Results ---")
print("Accuracy:", accuracy_score(y_test, lr_preds))
print("AUC:", roc_auc_score(y_test, lr_probs))
print("Confusion Matrix:\n", confusion_matrix(y_test, lr_preds))
print("Classification Report:\n", classification_report(y_test, lr_preds))

# Show most important features for Logistic Regression
# (display) the features that have the strongest influence on the model’s predictions
coef_df = pd.DataFrame({
    'Feature': columns[:-1],
    'Coefficient': lr_model.coef_[0]
})

coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)

print("\nTop 10 Most Important Features in Logistic Regression:")
print(coef_df[['Feature', 'Coefficient']].head(10))



--- Logistic Regression Results ---
Accuracy: 0.9346793349168646
AUC: 0.9768315923207228
Confusion Matrix:
 [[483  23]
 [ 32 304]]
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.95      0.95       506
           1       0.93      0.90      0.92       336

    accuracy                           0.93       842
   macro avg       0.93      0.93      0.93       842
weighted avg       0.93      0.93      0.93       842


Top 10 Most Important Features in Logistic Regression:
       Feature  Coefficient
26  feature_26    -4.015556
24  feature_24    -2.484120
45  feature_45    -1.336849
52  feature_52     1.228077
41  feature_41    -1.222367
40  feature_40    -1.189312
28  feature_28    -1.105883
6    feature_6     1.047907
55  feature_55     1.046090
25  feature_25    -0.985131


The Logistic Regression model performed very well, with an **accuracy of 93.47% and an AUC of 0.9768**, showing strong ability to distinguish between spam and non-spam emails.

The confusion matrix shows most emails were correctly classified, with only a few mistakes: **23 non-spam emails were labeled as spam and 32 spam emails were labeled as non-spam**.

The classification report tells us : <br>

For Precision:
* (0 - not spam) Out of all emails predicted as not spam, 94% were correct.
* (1- spam) Out of all emails the model predicted as spam, 93% were actually spam. <br>

For Recall:
* (0 - not spam) It correctly identified 95% of all non-spam emails. <br>
* (1- spam) The model correctly identified 90% of all actual spam emails <br>

The classification report shows high precision and recall for both classes, meaning the model is accurate and balanced.

The top features, such as feature_26, feature_24, and feature_52, had the biggest impact on the model's predictions. Overall, Logistic Regression proved to be a strong and reliable model for this task.

In [ ]:
# Model #2:  Naive Bayes
# Naive Bayes works well without scaling,
# so we keep the original feature values, GaussianNB
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

nb_preds = nb_model.predict(X_test)
nb_probs = nb_model.predict_proba(X_test)[:, 1]

print("\n--- Naive Bayes Results ---")
print("Accuracy:", accuracy_score(y_test, nb_preds))
print("AUC:", roc_auc_score(y_test, nb_probs))
print("Confusion Matrix:\n", confusion_matrix(y_test, nb_preds))
print("Classification Report:\n", classification_report(y_test, nb_preds))


--- Naive Bayes Results ---
Accuracy: 0.836104513064133
AUC: 0.9524721202710333
Confusion Matrix:
 [[384 122]
 [ 16 320]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.76      0.85       506
           1       0.72      0.95      0.82       336

    accuracy                           0.84       842
   macro avg       0.84      0.86      0.84       842
weighted avg       0.87      0.84      0.84       842



The Naive Bayes model achieved an **accuracy of 83.61% and an AUC of 0.9525**, showing good overall performance but lower than Logistic Regression.
From the confusion matrix:


* 384 non-spam emails were correctly classified
* 320 spam emails were correctly classified
* 122 non-spam emails were incorrectly labeled as spam
* 16 spam emails were incorrectly labeled as non-spam


This shows the model is very good at catching spam (high recall for spam), but it incorrectly labels more non-spam emails as spam. Overall, Naive Bayes performs well but is less balanced compared to Logistic Regression.

In [ ]:
# Cross-Validation for Logistic Regression
# Scale the full dataset for cross-validation
X_scaled = scaler.fit_transform(X)

cv_scores = cross_val_score(
    LogisticRegression(max_iter=5000),
    X_scaled,
    y,
    cv=5,
    scoring='accuracy'
)

print("\n--- Logistic Regression Cross-Validation ---")
print("Cross-validation accuracy scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())


--- Logistic Regression Cross-Validation ---
Cross-validation accuracy scores: [0.92042755 0.92874109 0.9239905  0.93467933 0.83491686]
Mean CV Accuracy: 0.9085510688836104


The cross-validation accuracy scores range from about **0.83 to 0.93, with a mean accuracy of 90.86%.**

This shows that the Logistic Regression model performs consistently well across different data splits. Although one score is slightly lower, the overall average remains high, indicating the model is reliable and generalizes well to new data.

**Conclusion**

In this assignment, the Spambase dataset was used to classify emails as spam or not spam. Both Logistic Regression and Naive Bayes models were applied and evaluated. While both performed well, **Logistic Regression achieved higher accuracy and more balanced results**. Naive Bayes was effective at detecting spam but made more errors when classifying non-spam emails. Overall, Logistic Regression proved to be the more reliable model for this task.